# TerraGen SDK Examples

This notebook demonstrates how to use TerraGen programmatically.

## Setup

First, make sure you have TerraGen installed and your API key configured.

In [ ]:
import os
import sys
from pathlib import Path

# Add terragen to path if running from examples folder
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Load environment variables
from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / '.env')

print(f"API Key configured: {'Yes' if os.environ.get('ANTHROPIC_API_KEY') else 'No'}")

## 1. Basic Generation

Generate Terraform code from a natural language prompt.

In [ ]:
from terragen.generator import generate_terraform
from terragen.config import PROVIDER_REGIONS
from pathlib import Path
import tempfile

# Create output directory
output_dir = Path(tempfile.mkdtemp(prefix='terragen_'))

# Generate S3 bucket with encryption
generate_terraform(
    prompt="S3 bucket with versioning and KMS encryption",
    output_dir=output_dir,
    provider="aws",
    region="us-east-1",
    chat_mode=False,
)

print(f"Generated files in: {output_dir}")
for f in output_dir.glob('*.tf'):
    print(f"  - {f.name}")

In [ ]:
# View generated main.tf
main_tf = output_dir / 'main.tf'
if main_tf.exists():
    print(main_tf.read_text())

## 2. Multi-Cloud Generation

Generate infrastructure for different cloud providers.

In [ ]:
# Check available providers and their default regions
for provider, config in PROVIDER_REGIONS.items():
    print(f"{provider.upper()}:")
    print(f"  Default region: {config['default']}")
    print(f"  Examples: {', '.join(config['examples'][:3])}")
    print()

In [ ]:
# Generate GCP infrastructure
gcp_output = Path(tempfile.mkdtemp(prefix='terragen_gcp_'))

generate_terraform(
    prompt="GCS bucket with uniform access and lifecycle policy",
    output_dir=gcp_output,
    provider="gcp",
    region="us-central1",
    chat_mode=False,
)

print("GCP main.tf:")
print((gcp_output / 'main.tf').read_text()[:1000])

## 3. Using the Agent Loop Directly

For more control, use the agent loop directly.

In [ ]:
from terragen.agent import run_agent
from terragen.config import SYSTEM_PROMPT

# Create working directory
work_dir = Path(tempfile.mkdtemp(prefix='terragen_agent_'))

# Custom prompt with specific requirements
prompt = f"""
Create an AWS VPC with:
- CIDR: 10.0.0.0/16
- 2 public subnets
- 2 private subnets
- NAT Gateway
- Internet Gateway
- Proper tagging with Environment=dev

Output directory: {work_dir}
"""

# Run the agent
result = run_agent(
    prompt=prompt,
    output_dir=work_dir,
    system_prompt=SYSTEM_PROMPT,
)

print("Agent completed!")
print(f"Files generated: {list(work_dir.glob('*.tf'))}")

## 4. Reading Existing Infrastructure

Use the modifier module to read and analyze existing Terraform code.

In [ ]:
from terragen.modifier import (
    read_terraform_files,
    read_state_file,
    summarize_state,
    get_git_info,
)

# Read terraform files from a directory
tf_files = read_terraform_files(output_dir)

print(f"Found {len(tf_files)} Terraform files:")
for name, content in tf_files.items():
    lines = len(content.split('\n'))
    print(f"  {name}: {lines} lines")

In [ ]:
# Check git info (if it's a repo)
git_info = get_git_info(output_dir)
print(f"Is git repo: {git_info['is_repo']}")
if git_info['is_repo']:
    print(f"Branch: {git_info['branch']}")
    print(f"Has uncommitted changes: {git_info['uncommitted_changes']}")

## 5. Clarifying Questions

Get service-specific clarifying questions for better generation.

In [ ]:
from terragen.questions import detect_service_type, get_questions_for_service, generate_clarifying_questions_llm

# Detect service type from prompt
prompts = [
    "S3 bucket for storing logs",
    "RDS PostgreSQL database",
    "EKS cluster with 3 nodes",
    "Lambda function with API Gateway",
]

for prompt in prompts:
    service = detect_service_type(prompt)
    print(f"'{prompt}' -> {service}")

In [ ]:
# Get LLM-generated clarifying questions (context-aware)
prompt = "EKS cluster with RDS PostgreSQL for a production API"
questions = generate_clarifying_questions_llm(prompt, provider="aws")

print(f"LLM-generated questions for: '{prompt}'")
for q in questions:
    print(f"\n  {q['question']}")
    if 'options' in q:
        print(f"    Options: {', '.join(q['options'][:4])}")
    if 'default' in q:
        print(f"    Default: {q['default']}")

## 6. Validation Tools

Validate generated Terraform code.

In [ ]:
from terragen.tools import execute_tool

# Run terraform fmt check
result = execute_tool(
    'run_command',
    {'command': f'cd {output_dir} && terraform fmt -check'},
    str(output_dir)
)
print(f"Format check: {'Passed' if 'error' not in result.lower() else 'Needs formatting'}")

In [ ]:
# Run terraform init and validate
import subprocess

# Init
init_result = subprocess.run(
    ['terraform', 'init', '-backend=false'],
    cwd=output_dir,
    capture_output=True,
    text=True
)
print(f"Init: {'Success' if init_result.returncode == 0 else 'Failed'}")

# Validate
validate_result = subprocess.run(
    ['terraform', 'validate', '-json'],
    cwd=output_dir,
    capture_output=True,
    text=True
)

import json
validation = json.loads(validate_result.stdout)
print(f"Validate: {'Valid' if validation.get('valid') else 'Invalid'}")

if not validation.get('valid'):
    for diag in validation.get('diagnostics', []):
        print(f"  - {diag.get('summary')}")

## 7. Cost Estimation

Estimate infrastructure costs using Infracost (if configured).

In [ ]:
import shutil

if shutil.which('infracost') and os.environ.get('INFRACOST_API_KEY'):
    cost_result = subprocess.run(
        ['infracost', 'breakdown', '--path', str(output_dir), '--format', 'json'],
        capture_output=True,
        text=True,
        env={**os.environ, 'INFRACOST_API_KEY': os.environ['INFRACOST_API_KEY']}
    )
    
    if cost_result.returncode == 0:
        cost_data = json.loads(cost_result.stdout)
        monthly_cost = cost_data.get('totalMonthlyCost', '0.00')
        print(f"Estimated monthly cost: ${monthly_cost}")
        
        # Show breakdown
        for project in cost_data.get('projects', []):
            for resource in project.get('breakdown', {}).get('resources', []):
                if float(resource.get('monthlyCost', 0) or 0) > 0:
                    print(f"  {resource['name']}: ${resource['monthlyCost']}")
    else:
        print(f"Infracost error: {cost_result.stderr}")
else:
    print("Infracost not available. Install it and set INFRACOST_API_KEY.")

## 8. Using the API Client

If running the web API, you can use HTTP requests.

In [ ]:
import requests

API_URL = "http://localhost:8000"

# Check if API is running
try:
    response = requests.get(f"{API_URL}/health", timeout=2)
    print(f"API Status: {response.json()}")
except requests.exceptions.ConnectionError:
    print("API not running. Start with: ./start.sh")

In [ ]:
# Example: Generate via API (requires authentication)
# This is for reference - you'd need a valid JWT token

example_request = {
    "prompt": "S3 bucket with versioning",
    "provider": "aws",
    "region": "us-east-1"
}

print("Example API request:")
print(json.dumps(example_request, indent=2))

# response = requests.post(
#     f"{API_URL}/generate/",
#     json=example_request,
#     headers={"Authorization": f"Bearer {token}"}
# )

## 9. Cleanup

In [ ]:
import shutil

# Clean up temporary directories
for d in [output_dir, gcp_output, work_dir]:
    if d.exists():
        shutil.rmtree(d)
        print(f"Cleaned up: {d}")

## Summary

TerraGen SDK provides:

| Module | Purpose |
|--------|--------|
| `terragen.generator` | High-level `generate_terraform()` |
| `terragen.agent` | Low-level `run_agent()`, `run_interactive_session()` |
| `terragen.modifier` | Read/modify existing Terraform |
| `terragen.questions` | Clarifying questions - `generate_clarifying_questions_llm()` |
| `terragen.tools` | Tool execution (write_file, read_file, run_command) |
| `terragen.config` | Configuration, prompts, constants |